# Question 3: WebText API proof of concept (Wikipedia: Natural language processing)

**Analytics flow**
1. **3.1** Scrape the article's paragraphs into a data frame.
2. **3.2** Pre-process the text.
3. **3.3** Two extraction techniques: **TF-IDF** keywords and **TextRank** extractive summary sentences, each with a chart.

The page is fetched live, so the exact numbers can differ slightly from the recorded screenshots if the article has been edited.

In [ ]:
library(rvest)
library(tidyverse)
library(tm)
library(tidytext)
library(textrank)
library(ggplot2)

## 3.1 Web scraping

In [ ]:
url <- "https://en.wikipedia.org/wiki/Natural_language_processing"
webpage <- read_html(url)

paragraphs <- webpage %>%
  html_nodes("p") %>%
  html_text()

paragraphs <- paragraphs[paragraphs != "" & nchar(paragraphs) > 20]
full_text  <- paste(paragraphs, collapse = " ")

web_df <- data.frame(file_name = url, content = full_text, stringsAsFactors = FALSE)
str(web_df)

**Recorded output** (screenshot from the project documentation): scraped data frame

![scraped data frame](../docs/images/q3_1_scrape.png)

## 3.2 Text pre-processing

In [ ]:
corpus <- VCorpus(VectorSource(web_df$content))
corpus <- tm_map(corpus, content_transformer(tolower))
corpus <- tm_map(corpus, removePunctuation)
corpus <- tm_map(corpus, removeNumbers)
corpus <- tm_map(corpus, removeWords, stopwords("english"))
corpus <- tm_map(corpus, stripWhitespace)

cleaned_text <- content(corpus[[1]])
print("Text preprocessing complete (first 200 chars)")
print(substr(cleaned_text, 1, 200))

**Recorded output** (screenshot from the project documentation): cleaned text preview

![cleaned text preview](../docs/images/q3_2_preprocess.png)

## 3.3 Information extraction

### Technique 1: TF-IDF keyword extraction
Each paragraph is treated as a document, so a word scores highly when it is frequent in a paragraph but rare across the others.

In [ ]:
para_df <- tibble(doc_id = seq_along(paragraphs), text = paragraphs)

para_tokens <- para_df %>%
  unnest_tokens(word, text) %>%
  anti_join(stop_words, by = "word") %>%
  count(doc_id, word, sort = TRUE)

total_words <- para_tokens %>%
  group_by(doc_id) %>%
  summarise(total = sum(n), .groups = "drop")

para_tokens <- left_join(para_tokens, total_words, by = "doc_id")
para_tfidf <- para_tokens %>%
  bind_tf_idf(word, doc_id, n) %>%
  arrange(desc(tf_idf))

print("--- Top TF-IDF keywords preview ---")
print(head(para_tfidf, 5))

**Recorded output** (screenshot from the project documentation): top TF-IDF terms

![top TF-IDF terms](../docs/images/q3_3_tfidf.png)

### Technique 2: TextRank extractive summary
Sentences (here, paragraphs) are nodes in a graph linked by shared words; the highest-ranked ones form the summary.

In [ ]:
sentence_words <- para_df %>%
  unnest_tokens(word, text) %>%
  anti_join(stop_words, by = "word")

tr_sentences <- textrank_sentences(data = para_df, terminology = sentence_words)
summary_sentences <- summary(tr_sentences, n = 3, sentence_format = "markdown")

print("--- Extractive summary (top sentences via TextRank) ---")
print(summary_sentences)

**Recorded output** (screenshot from the project documentation): TextRank summary

![TextRank summary](../docs/images/q3_3_textrank.png)

### Charts

In [ ]:
top_keywords <- para_tfidf %>%
  group_by(word) %>%
  summarise(mean_tf_idf = mean(tf_idf), .groups = "drop") %>%
  slice_max(mean_tf_idf, n = 10)

vis_1 <- ggplot(top_keywords, aes(x = reorder(word, mean_tf_idf), y = mean_tf_idf, fill = mean_tf_idf)) +
  geom_col(show.legend = FALSE) +
  coord_flip() +
  labs(title = "Top 10 Keywords by TF-IDF Score",
       subtitle = "WebText API Proof of Concept - Content Analysis",
       x = "Keywords", y = "Mean TF-IDF Score") +
  theme_minimal()

print(vis_1)

**Recorded output** (screenshot from the project documentation): top 10 keywords by TF-IDF

![top 10 keywords by TF-IDF](../docs/images/q3_3_tfidf_chart.png)

In [ ]:
top_sent_scores <- head(tr_sentences$sentences, 5) %>%
  mutate(sentence_id = row_number())

vis_2 <- ggplot(top_sent_scores, aes(x = reorder(as.factor(sentence_id), textrank), y = textrank, fill = textrank)) +
  geom_col(show.legend = FALSE) +
  coord_flip() +
  labs(title = "Top Extractive Sentences by TextRank Score",
       subtitle = "Graph-Based Sentence Importance Ranking",
       x = "Sentence ID", y = "TextRank Score") +
  theme_minimal()

print(vis_2)

**Recorded output** (screenshot from the project documentation): TextRank sentence scores

![TextRank sentence scores](../docs/images/q3_3_textrank_chart.png)

*Changes from `src/q3_webtext_api.R`:* `data_frame()` (deprecated in tibble) is now `tibble()`, and the stray `stringsAsFactors` argument, which would have added a column, was removed.